# LAB 5: Implementing Truyen Kieu's poem style generator using GPT-2 (fine-tuned for Vietnamese)

## 1. Introduction

This lab demonstrates how to leverage GPT-2 (fine-tuned for Vietnamese) for poem generation. We will use nlpHUST's GPT-2 for our model, and Truyen Kieu's raw dataset as our dataset for fine-tuning. Our goal is to create a model that can generate poems which are similar to Truyen Kieu's style.

In [1]:
# %pip install torch transformers
# %pip install transformers[torch]
# %pip install accelerate>=0.26.0
# %pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

## 2. Idea

### 2.1. Fine-tuning GPT-2 for Vietnamese Poetry Generation


The core idea is to fine-tune a pre-trained GPT-2 model on classical Vietnamese poetry, specifically "Truyện Kiều" by Nguyễn Du, to generate new poems that capture the style, rhythm, and linguistic patterns of this masterpiece.

**Example Process:**

**Input text from Truyện Kiều:**
> "Trăm năm trong cõi người ta,  
> Chữ tài chữ mệnh khéo là ghét nhau"

**Expected generated output:**
> "Nghìn thu trong kiếp phong ba,  
> Tình duyên số phận nào là bền lâu"

### 2.2. Data preprocessing


As a popular saying: "Garbage in, garbage out", data preprocessing is important. We can not just take the Truyen Kieu's raw dataset and feed it to fine-tuning our model. The output will be extremely awful, as shown in this example:

In [ ]:
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel

# Load Vietnamese GPT-2 model
tokenizer = GPT2Tokenizer.from_pretrained('NlpHUST/gpt2-vietnamese')
model = GPT2LMHeadModel.from_pretrained('NlpHUST/gpt2-vietnamese')

raw_text = "Trăm năm trong cõi người ta, Chữ tài chữ mệnh khéo là ghét nhau. Trải qua một cuộc bể dâu"

# Generate with raw data
input_ids = tokenizer.encode(raw_text, return_tensors='pt')

print("Raw text input:")
print(f"Input: {raw_text}")

with torch.no_grad():
    output = model.generate(
        input_ids,
        max_length=input_ids.shape[1] + 50,
        temperature=0.8,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
print(f"Output: {generated_text[len(raw_text):]}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

... (2 outputs left out)

pytorch_model.bin:   0%|          | 0.00/510M [00:00<?, ?B/s]

Raw text input:
Input: Trăm năm trong cõi người ta, Chữ tài chữ mệnh khéo là ghét nhau. Trải qua một cuộc bể dâu


model.safetensors:   0%|          | 0.00/510M [00:00<?, ?B/s]

Output: , ngày nay ta đã nhận thấy ở người xưa các phương châm của Khổng Mạnh vẫn được người đời sử dụng như các phương châm “đạo và tài” và các phương châm “đức” (đức và tài) vẫn được người đời sử dụng và

Notice: The output lacks proper poetic structure, rhythm, and style!


Therefore, we should workout someway of preprocessing that can capture all the aspects of Truyen Kieu poem. Some approaches for this should be:
-  Normalize the text
-  Truyen Kieu has a "Lục Bát" pattern, therefore a markup should be make to keep track with the verse.
-  Detect rhyme patterns.


#### 2.2.1. Normalize and coupling the text
Apart from standard normalization methods, which you should have known by now, one approach is that an implement should be made in the normalization process so that the lines will not storage separately but storage in a sample, with is a couple of a Lục (6) line and a Bát (8) line.

This approach will make the the model learn the structural of the Lục bát poem and the verbal pattern in each sample. 

#### 2.2.2. Mark-up with speicial tokens

Another approach is to provide speicial tokens to mark the structural of the poem. Speicial tokens like
```
<poem>
<luc>
<bat>
</poem>
```
The tokens help the model to predict the next tokens and where to begin/end a Lục (6) or a Bát (8) token.

In [ ]:
# Build (lục, bát) couplets from the raw dataset
import re

def normalize(line: str) -> str:
    line = re.sub(r"[^\wÀ-ỹ\s'-]+", " ", line)
    return re.sub(r"\s+", " ", line).strip()

with open("dataset/truyenkieu.txt", encoding="utf-8") as f:
    lines = [normalize(l) for l in f if l.strip()]

couplets = []
i = 0
while i < len(lines) - 1:
    if len(lines[i].split()) == 6 and len(lines[i + 1].split()) == 8:
        couplets.append((lines[i], lines[i + 1]))
        i += 2
    else:
        i += 1
print(f"{len(couplets)} couplets, e.g. {couplets[0]}")


In [ ]:
def format_sample(luc: str, bat: str) -> str:
    return f"<poem>\n<luc> {luc} </luc>\n<bat> {bat} </bat>\n</poem>"

samples = [format_sample(l, b) for (l, b) in couplets[:5]]
print(samples[0])

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

# Add special tokens for poem structure
special = {"additional_special_tokens": ["<poem>", "</poem>", "<luc>", "</luc>", "<bat>", "</bat>"]}
tokenizer.add_special_tokens(special)

#### 2.2.3. Tailoring the prompt

Prompt can be tailored to guide the model to favored pattern or activate its pattern recognization (this could be handly with speicial tokens approach)

In [ ]:
import torch
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("gpt2")
model.resize_token_embeddings(len(tokenizer))

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

@torch.no_grad()
def generate_couplet(prompt="<poem>\n<luc>", max_new_tokens=80):
    model.eval()
    inp = tokenizer(prompt, return_tensors="pt").to(device)
    out = model.generate(
        **inp,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.9,
        top_p=0.95,
        repetition_penalty=1.15,
        pad_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(out[0], skip_special_tokens=False)

print(generate_couplet())
